In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml
from pyspark.sql import functions as F

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from riesgo_credito.procesamiento import esquema, etl

TRIMESTRE = "2019Q1"
DATOS = RAIZ / "data"
PARTICION = f"trimestre={TRIMESTRE}"
RUTAS = {
    "originacion": DATOS / "limpios" / "originacion" / PARTICION,
    "desempeno": DATOS / "limpios" / "desempeno" / PARTICION,
    "p1": DATOS / "modelado" / "p1" / PARTICION,
    "p2": DATOS / "modelado" / "p2" / PARTICION,
    "tablero_originacion": DATOS / "tablero" / "originacion" / PARTICION,
    "tablero_serie_mensual": DATOS / "tablero" / "serie_mensual" / PARTICION,
    "tablero_cartera_activa": DATOS / "tablero" / "cartera_activa" / PARTICION,
}

params = yaml.safe_load((RAIZ / "params.yaml").read_text())
spark = etl.crear_sesion(params["spark"]["memoria_driver"], params["spark"]["particiones"])
spark.sparkContext.setLogLevel("ERROR")
leer = {n: spark.read.parquet(str(r)) for n, r in RUTAS.items()}

metricas = json.loads((DATOS / "metricas" / "etl" / f"{TRIMESTRE}.json").read_text())
orig = pd.read_parquet(RUTAS["originacion"])
p1 = pd.read_parquet(RUTAS["p1"])
pd.Series(metricas, name="valor").to_frame()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/29 04:16:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


,valor
trimestre,2019Q1
creditos,279861
registros_desempeno,9821276
periodo_inicial,201901
fecha_corte,202603
excluidos_codigo_96,801
p1_creditos,279060
p1_etiquetables,279060
p1_positivos,680
p2_filas,9399814


## Cifras de referencia del EDA

In [2]:
referencia = {
    "creditos": 279861, "registros_desempeno": 9821276, "excluidos_codigo_96": 801,
    "p1_etiquetables": 279060, "p1_positivos": 680,
    "p2_etiquetables": 9129854, "p2_positivos": 14938,
    "p2_30_etiquetables": 9236499, "p2_30_positivos": 35421,
    "p2_60_etiquetables": 9129930, "p2_60_positivos": 42799,
    "activos_al_corte": 55339,
}
comparacion = pd.DataFrame({"eda": referencia, "etl": {k: metricas[k] for k in referencia}})
comparacion["coincide"] = comparacion.eda == comparacion.etl
assert comparacion.coincide.all()
comparacion

,eda,etl,coincide
creditos,279861,279861,True
registros_desempeno,9821276,9821276,True
excluidos_codigo_96,801,801,True
p1_etiquetables,279060,279060,True
p1_positivos,680,680,True
p2_etiquetables,9129854,9129854,True
p2_positivos,14938,14938,True
p2_30_etiquetables,9236499,9236499,True
p2_30_positivos,35421,35421,True
p2_60_etiquetables,9129930,9129930,True


El ETL en Spark reproduce exactamente las cifras del EDA en las dos preguntas, incluidas las dos variantes de la etiqueta de la pregunta 2 y los créditos activos al corte. La lectura, la limpieza, la exclusión del código 96 y las etiquetas quedaron bien portadas.

## Calidad de las salidas

In [3]:
perf, p2 = leer["desempeno"], leer["p2"]
sin_defecto = set(orig.loc[~orig.defecto, "loan_id"])
defecto = spark.createDataFrame(orig.loc[orig.defecto, ["loan_id"]])
chequeos = pd.Series({
    "duplicados en originación": orig.loan_id.duplicated().sum(),
    "duplicados en desempeño (crédito-mes)": perf.count() - perf.select("loan_id", "period").distinct().count(),
    "duplicados en P1": p1.loan_id.duplicated().sum(),
    "duplicados en P2 (crédito-mes)": p2.count() - p2.select("loan_id", "period").distinct().count(),
    "créditos sin defecto que faltan en P1": len(sin_defecto - set(p1.loan_id)),
    "créditos con código 96 en P1": len(set(p1.loan_id) - sin_defecto),
    "créditos con código 96 en P2": p2.join(defecto, "loan_id").select("loan_id").distinct().count(),
    "meses de P2 con atraso distinto de 0 o 30 días": p2.where(~F.col("dlq_num").isin(0, 1)).count(),
    "columnas posteriores al cierre en desempeño o P2": len(set(esquema.POSTERIORES_AL_CIERRE) & set(perf.columns + p2.columns)),
    "columnas fuera de la solicitud en P1": len(set(p1.columns) - set(etl.VARIABLES_P1) - {"loan_id", "first_payment_date", "y_p1", "trimestre"}),
    "puntajes 9999 en P1": (p1.credit_score == 9999).sum(),
    "deuda-ingreso 999 en P1": (p1.dti == 999).sum(),
    "registros previos al primer pago (edad <= 0)": perf.where(F.col("edad") <= 0).count(),
}, name="valor")
chequeos.to_frame()

,valor
duplicados en originación,0
duplicados en desempeño (crédito-mes),0
duplicados en P1,0
duplicados en P2 (crédito-mes),0
créditos sin defecto que faltan en P1,0
créditos con código 96 en P1,0
créditos con código 96 en P2,0
meses de P2 con atraso distinto de 0 o 30 días,0
columnas posteriores al cierre en desempeño o P2,0
columnas fuera de la solicitud en P1,0


In [4]:
n_p2 = p2.count()
nulos_p2 = p2.select([(100 * F.count(F.when(F.col(c).isNull(), 1)) / n_p2).alias(c) for c in p2.columns]).toPandas().iloc[0]
nulos = pd.concat([(100 * p1.isna().mean()).rename("P1"), nulos_p2.rename("P2")], axis=1).round(3)
nulos[(nulos.fillna(0) > 0).any(axis=1)]

,P1,P2
credit_score,0.054,NaN
dti,0.400,NaN
ocltv,0.003,NaN
eltv,NaN,10.615
meses_desde_atraso,NaN,88.657
plan_asistencia,NaN,99.498
y_p2,NaN,1.737
y_p2_60,NaN,1.733


Las llaves son únicas en todas las tablas, ninguna tabla de modelado tiene créditos con código 96 ni columnas posteriores al cierre, y P1 solo tiene variables de la solicitud. En P2, los nulos de meses_desde_atraso son créditos que nunca se han atrasado, los de plan_asistencia son meses sin plan y los de las etiquetas son los últimos meses de los créditos activos, cuya ventana de tres meses aún no se alcanza a observar. Los registros con edad cero o negativa son los meses previos al primer pago y se conservan en la capa limpia, como en el EDA.

## Pregunta 2: señal de las variables

In [5]:
carta = p2.where((F.col("dlq_num") == 0) & F.col("y_p2").isNotNull())
atrasos = F.when(F.col("atrasos_12m") == 0, "0").when(F.col("atrasos_12m") == 1, "1") \
    .when(F.col("atrasos_12m") <= 3, "2-3").otherwise("4+")
por_atrasos = carta.groupBy(atrasos.alias("atrasos_12m")).agg(
    F.count("*").alias("creditos_mes"), (100 * F.avg("y_p2")).alias("tasa_pct")).toPandas()
por_atrasos["tasa_eda_pct"] = por_atrasos.atrasos_12m.map({"0": 0.123, "1": 0.490, "2-3": 0.905, "4+": 1.795})
por_atrasos.sort_values("atrasos_12m").round(3)

,atrasos_12m,creditos_mes,tasa_pct,tasa_eda_pct
3,0,8704916,0.123,0.123
0,1,190545,0.490,0.490
2,2-3,103924,0.905,0.905
1,4+,130469,1.795,1.795


In [6]:
def auc(df, columna, decimales=None):
    valor = F.round(columna, decimales) if decimales is not None else F.col(columna).cast("double")
    t = (df.where(F.col(columna).isNotNull()).groupBy(valor.alias("v"))
         .agg(F.sum("y_p2").alias("pos"), (F.count("*") - F.sum("y_p2")).alias("neg"))
         .toPandas().sort_values("v"))
    negativos_menores = t.neg.cumsum() - t.neg
    return (t.pos * (negativos_menores + 0.5 * t.neg)).sum() / (t.pos.sum() * t.neg.sum())


carta = carta.withColumn("plan_actual", F.col("plan_asistencia").isNotNull())
variables = {"meses_desde_atraso": None, "en_pandemia": None, "eltv": None, "pct_saldo": 4, "atrasos_12m": None,
             "max_atraso_12m": None, "indulgencia_12m": None, "plan_actual": None, "diferimiento": None,
             "saldo_sin_interes": None, "mora_desastre": None, "modificado": None, "dif_tasa": 3, "edad": None}
filas = []
for v, d in variables.items():
    a = auc(carta, v, d)
    filas.append({"variable": v, "auc": max(a, 1 - a), "sentido": "+" if a >= 0.5 else "-"})
pd.DataFrame(filas).sort_values("auc", ascending=False).round(3)

,variable,auc,sentido
0,meses_desde_atraso,0.799,-
1,en_pandemia,0.717,+
2,eltv,0.627,+
3,pct_saldo,0.622,+
4,atrasos_12m,0.620,+
5,max_atraso_12m,0.620,+
6,indulgencia_12m,0.554,+
9,saldo_sin_interes,0.547,+
7,plan_actual,0.546,+
8,diferimiento,0.544,+


Las tasas por atrasos previos coinciden con las del EDA. Con todos los datos, la recencia del último atraso es la señal más fuerte entre los créditos que alguna vez se atrasaron, seguida del préstamo-valor actual, la proporción de saldo pendiente y los atrasos recientes. Las variables de asistencia y de modificación aportan poco por sí solas. La pandemia separa más que casi todas, lo que confirma que distorsiona la etiqueta y que su tratamiento debe decidirse en los experimentos.

## Tablas del tablero

In [7]:
t_orig = pd.read_parquet(RUTAS["tablero_originacion"])
t_serie = pd.read_parquet(RUTAS["tablero_serie_mensual"])
t_cartera = leer["tablero_cartera_activa"]
serie = t_serie.groupby("period")[["activos", "mora_90"]].sum()
serie["pct_mora_90"] = 100 * serie.mora_90 / serie.activos
pd.Series({
    "créditos en la tabla de originación": t_orig.creditos.sum(),
    "etiquetables de P1 en la tabla de originación": t_orig.p1_etiquetables.sum(),
    "positivos de P1 en la tabla de originación": t_orig.p1_positivos.sum(),
    "activos en la fecha de corte": serie.loc[metricas["fecha_corte"], "activos"],
    "mes con más mora de 90 días": serie.pct_mora_90.idxmax(),
    "máxima mora de 90 días (% de activos)": round(serie.pct_mora_90.max(), 2),
    "créditos en la cartera activa": t_cartera.count(),
    "créditos de la cartera activa que no están al día": t_cartera.where(F.col("dlq_num") != 0).count(),
}, name="valor", dtype=object).to_frame()

,valor
créditos en la tabla de originación,279861
etiquetables de P1 en la tabla de originación,279060
positivos de P1 en la tabla de originación,680
activos en la fecha de corte,55339
mes con más mora de 90 días,202102
máxima mora de 90 días (% de activos),4.48
créditos en la cartera activa,53605
créditos de la cartera activa que no están al día,0


La tabla de originación conserva todos los créditos y los positivos de P1, la serie mensual reproduce los activos al corte y el pico de mora de 90 días del EDA, y la cartera activa solo tiene créditos al día en la fecha de corte.

## Tablas generadas

In [8]:
resumen = pd.DataFrame({
    n: {"filas": df.count(), "columnas": len(df.columns),
        "mb": sum(f.stat().st_size for f in RUTAS[n].glob("*.parquet")) / 1e6}
    for n, df in leer.items()
}).T
spark.stop()
resumen.astype({"filas": int, "columnas": int}).round(1)

,filas,columnas,mb
originacion,279861,27,5.0
desempeno,9821276,25,126.4
p1,279060,19,4.0
p2,9399814,23,137.5
tablero_originacion,4056,16,0.2
tablero_serie_mensual,4692,14,0.2
tablero_cartera_activa,53605,37,2.0
